In [ ]:
# Install the Ultralytics package which provides the YOLOv8 framework.
# Used for auto-annotation during ETL and for training the final model.
!pip install ultralytics -q

In [ ]:
import cv2
from collections import deque
from ultralytics import YOLO

def final_production_test(video_path, model_path):
    """Run the trained surveillance model against a video and return a final verdict.

    Uses the same sliding-window logic as the live API pipeline to validate
    that the newly trained model behaves correctly on unseen footage before
    deploying it to the Flask inference server.

    The function prints a per-frame log so you can visually verify that
    detections are sustained (not just a single noisy frame) before the
    final verdict is issued.

    Args:
        video_path (str): Path to the test video file.
        model_path (str): Path to the trained .pt weights file.

    Returns:
        str: One of 'ACCIDENT', 'FIGHT', or 'NORMAL'.
    """
    print(f"Running Final Production Analysis on: {video_path}...\n")
    
    model = YOLO(model_path)
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    
    # Detection gate parameters — must match the values used in the live API
    # so test results accurately predict production behaviour.
    CONFIDENCE_TARGET = 0.60    # Minimum YOLO confidence to accept a detection.
    WINDOW_SIZE = 30            # Rolling window length in frames.
    # An incident is only confirmed when at least 15 of the last 30 frames
    # contain a positive detection — prevents single-frame false positives.
    ACTIVATION_THRESHOLD = 15
    
    # Two independent deques act as sliding windows, one per incident class.
    # Each frame appends 1 (detected) or 0 (not detected); the deque
    # automatically discards the oldest entry once maxlen is reached.
    recent_accidents = deque(maxlen=WINDOW_SIZE)
    recent_fights = deque(maxlen=WINDOW_SIZE)
    
    frame_count = 0
    final_verdict = "NORMAL"
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break 
            
        # Convert frame index to wall-clock seconds for the per-frame log.
        timestamp = frame_count / fps if fps > 0 else 0
            
        # Run inference — verbose=False suppresses YOLO's per-frame console output.
        results = model.predict(frame, conf=CONFIDENCE_TARGET, verbose=False)
        frame_classes = [model.names[int(box.cls[0].item())] for box in results[0].boxes]
            
        is_acc = 'accident' in frame_classes
        is_fgt = 'fight' in frame_classes
            
        status_text = "ACCIDENT" if is_acc else ("FIGHT" if is_fgt else "NORMAL")
        print(f"Frame {frame_count:04d} | Time: {timestamp:.2f}s | Detection: {status_text}")
        
        # Update rolling windows with this frame's binary signal.
        recent_accidents.append(1 if is_acc else 0)
        recent_fights.append(1 if is_fgt else 0)
        
        # Only update the verdict once — the first confirmed incident wins.
        # Once set, ignore subsequent frames (early exit on next iteration).
        if final_verdict == "NORMAL":
            if sum(recent_accidents) >= ACTIVATION_THRESHOLD:
                final_verdict = "ACCIDENT"
            elif sum(recent_fights) >= ACTIVATION_THRESHOLD:
                final_verdict = "FIGHT"

        frame_count += 1

    cap.release()
    return final_verdict

# Point to the freshly trained model weights and a test video.
# This path matches the output of the training cell below.
new_model_path = '/kaggle/working/runs/detect/Surveillance_System/purified_tracker/weights/best.pt'
my_video = '/kaggle/input/roadsurveillance-testdata/accident.mp4' 

result = final_production_test(my_video, new_model_path)

# Print the final verdict prominently so it is easy to spot in long output logs.
print("\n" + "="*45)
print(f" ULTIMATE VERDICT FOR NORMAL VIDEO: {result}")
print("="*45)

In [ ]:
import os
import shutil
import glob
import yaml
import cv2
from ultralytics import YOLO

def hybrid_etl_pipeline(dataset_paths, output_dir='clean_surveillance_dataset', max_auto_images=2500):
    """Build a clean, unified YOLO-formatted training dataset from multiple Kaggle sources.

    This is a 'hybrid' pipeline because it combines two annotation strategies:

      PATH A — Human-verified labels:
        If a .txt annotation file already exists alongside or parallel to the
        image, copy it directly. Only the bounding-box coordinates are kept;
        the original class ID is replaced with the unified class ID so all
        source datasets share a consistent numbering scheme.

      PATH B — YOLOv8n auto-annotation:
        If no label exists, run a lightweight YOLOv8n model to generate one
        automatically. This path is intentionally BLOCKED for the 'accident'
        class because auto-annotating accident images produced noisy,
        low-quality labels that hurt model accuracy in earlier experiments.
        Auto-annotation is capped at max_auto_images per class to stay within
        Kaggle's session time limit.

    Output directory structure (required by YOLOv8 trainer):
        <output_dir>/
            images/train/     <- all training images
            labels/train/     <- matching YOLO .txt annotation files
            data.yaml         <- class list and path config for the trainer

    Args:
        dataset_paths  (dict): Maps class name -> list of image directory paths.
                               Dict insertion order determines numeric class IDs
                               (0, 1, 2, ...) written into label files.
        output_dir     (str):  Root directory where the merged dataset is written.
        max_auto_images (int): Maximum images to auto-annotate per class.

    Returns:
        str: Path to the generated data.yaml file.
    """
    print("🧹 Starting HYBRID Data Extraction...\n")
    
    # Create the exact directory layout YOLOv8 expects for training.
    img_dir = os.path.join(output_dir, 'images', 'train')
    lbl_dir = os.path.join(output_dir, 'labels', 'train')
    os.makedirs(img_dir, exist_ok=True)
    os.makedirs(lbl_dir, exist_ok=True)

    # Load a lightweight YOLOv8n model used only as the auto-annotator.
    # YOLOv8n is fast enough to process thousands of images within Kaggle's
    # session time limit while still producing usable bounding boxes.
    print("Loading base annotator for missing labels...")
    annotator = YOLO('yolov8n.pt') 
    
    # Track how many images came from each source to help diagnose class imbalance.
    class_counts = {'human_verified': {}, 'auto_annotated': {}}

    for class_id, (class_name, paths) in enumerate(dataset_paths.items()):
        print(f"\nProcessing '{class_name}' (ID: {class_id})...")
        human_count = 0
        auto_count = 0
        
        for path in paths:
            if not os.path.exists(path):
                continue
                
            # Recursively collect all JPEG and PNG files under this directory.
            files = glob.glob(os.path.join(path, '**', '*.[jp][pn]*[g]'), recursive=True) 
            
            for f in files:
                # Search for an existing label in two standard locations:
                # 1. Same directory as the image, .txt extension.
                # 2. Sibling 'labels/' folder (common in YOLO dataset layouts).
                txt_side = os.path.splitext(f)[0] + '.txt'
                txt_parallel = f.replace('/images/', '/labels/')
                txt_parallel = os.path.splitext(txt_parallel)[0] + '.txt'
                found_txt = txt_side if os.path.exists(txt_side) else (txt_parallel if '/images/' in f and os.path.exists(txt_parallel) else None)

                # Use a combined counter as part of the output filename to
                # guarantee uniqueness even when multiple source datasets
                # contain images with identical base filenames.
                out_name = f"{class_name}_{human_count + auto_count}"
                img_dest = os.path.join(img_dir, f"{out_name}.jpg")
                lbl_dest = os.path.join(lbl_dir, f"{out_name}.txt")

                if found_txt:
                    # ── PATH A: Human-verified label exists ──────────────────
                    # Copy the image and re-write the label using the unified
                    # class ID. Columns 1-4 (cx, cy, w, h) are preserved exactly;
                    # column 0 (original class ID) is replaced with class_id.
                    shutil.copy(f, img_dest)
                    with open(found_txt, 'r') as in_txt, open(lbl_dest, 'w') as out_txt:
                        for line in in_txt:
                            parts = line.strip().split()
                            if len(parts) >= 5:
                                out_txt.write(f"{class_id} {parts[1]} {parts[2]} {parts[3]} {parts[4]}\n")
                    human_count += 1
                    
                else:
                    # ── PATH B: No label — attempt auto-annotation ───────────
                    # STRICT RULE: Never auto-annotate accident images.
                    # In earlier training runs, auto-generated accident labels
                    # introduced noisy bounding boxes that caused the model to
                    # over-detect accidents on ordinary street scenes.
                    if class_name == 'accident':
                        continue 
                        
                    # Enforce the per-class auto-annotation cap to keep total
                    # processing time within Kaggle's 12-hour session limit.
                    if auto_count >= max_auto_images:
                        continue
                        
                    frame = cv2.imread(f)
                    if frame is not None:
                        # Detect only relevant COCO classes:
                        # person(0), car(2), motorcycle(3), bus(5), truck(7).
                        # Filtering classes speeds up inference and avoids
                        # saving images that contain only irrelevant objects.
                        results = annotator.predict(frame, classes=[0, 2, 3, 5, 7], conf=0.5, verbose=False)
                        
                        # Only save the image if at least one target object was found —
                        # images with empty labels degrade training.
                        if len(results[0].boxes) > 0:
                            cv2.imwrite(img_dest, frame)
                            # Write normalised YOLO format: class_id cx cy w h
                            with open(lbl_dest, 'w') as txt_f:
                                for box in results[0].boxes:
                                    x_c, y_c, w, h = box.xywhn[0].tolist()
                                    txt_f.write(f"{class_id} {x_c:.6f} {y_c:.6f} {w:.6f} {h:.6f}\n")
                            auto_count += 1
                            
        class_counts['human_verified'][class_name] = human_count
        class_counts['auto_annotated'][class_name] = auto_count
        print(f"   Saved {human_count} human labels |  Auto-annotated {auto_count} images.")

    # Generate data.yaml — the config file that tells the YOLO trainer where
    # the images are and what the class names are.
    # val points to the same folder as train because this notebook focuses on
    # dataset assembly; a proper train/val split can be done separately.
    yaml_data = {
        'train': 'images/train',
        'val': 'images/train',  # Reusing train as val — split externally if needed.
        'nc': len(dataset_paths),
        'names': list(dataset_paths.keys())
    }
    yaml_path = os.path.join(output_dir, 'data.yaml')
    with open(yaml_path, 'w') as f:
        yaml.dump(yaml_data, f)
        
    print(f"\n ETL Complete! Dataset ready at {yaml_path}")
    return yaml_path


# Dataset paths — each key becomes a class name; dict order sets the class ID.
# 'accident' images come ONLY from human-annotated sources (PATH B is blocked).
# 'normal', 'fight', and 'vehicle' may use auto-annotation to fill gaps.
my_datasets = {
    # Class 0: confirmed road accident frames from a dedicated YOLO dataset.
    'accident': [
        '/kaggle/input/datasets/amedeograndi/accidents-detection-dataset/AccidentsDetectionYOLOv8/train/images',
        '/kaggle/input/datasets/amedeograndi/accidents-detection-dataset/AccidentsDetectionYOLOv8/valid/images',
        '/kaggle/input/datasets/amedeograndi/accidents-detection-dataset/AccidentsDetectionYOLOv8/test/images'
    ],
    # Class 1: non-violent scenes used as negative/background examples.
    'normal': [
        '/kaggle/input/datasets/karandeep98/real-life-violence-and-nonviolence-data/violence_dataset/non_violence'
    ],
    # Class 2: real-life violence/fight footage.
    'fight': [
        '/kaggle/input/datasets/karandeep98/real-life-violence-and-nonviolence-data/violence_dataset/violence'
    ],
    # Class 3: general vehicle images from COCO 2017 to improve vehicle localisation.
    'vehicle': [
        '/kaggle/input/datasets/sabahesaraki/2017-2017/train2017/train2017',
        '/kaggle/input/datasets/sabahesaraki/2017-2017/val2017/val2017'
    ]
}

# Run the ETL pipeline. max_auto_images=5000 allows more auto-annotated images
# than the function default while still fitting within Kaggle's GPU time limit.
yaml_file = hybrid_etl_pipeline(my_datasets, max_auto_images=5000)

In [ ]:
from ultralytics import YOLO

# Initialise YOLOv8 Nano from COCO pre-trained weights.
# Starting from pre-trained weights (transfer learning) dramatically reduces
# the number of epochs needed to reach good accuracy on this domain-specific
# dataset compared to training from random initialisation.
print("Initializing fresh YOLOv8 Nano model...")
model = YOLO('yolov8n.pt') 

print("\nTraining on purified data...")
results = model.train(
    data='/kaggle/working/clean_surveillance_dataset/data.yaml',
    epochs=30,   # 30 epochs is enough for a clean dataset 
    imgsz=640,   # standard YOLO input resolution; balances speed and accuracy.           
    batch=16,    # moderate batch size               
    project='Surveillance_System',  #results, plots and weights are saved under: /kaggle/working/runs/detect/Surveillance_System/purified_tracker/
    name='purified_tracker'
)

# The trainer saves two checkpoints: last.pt (final epoch) and best.pt
# (highest validation mAP). best.pt is what gets deployed to the Flask server.
print("\n Training Complete! Your new model is located at:")
print("/kaggle/working/runs/detect/Surveillance_System/purified_tracker/weights/best.pt")

In [ ]:
# api-connector
# This cell runs the complete 3-phase surveillance pipeline end-to-end
# and posts the results to the Django backend via its REST API.
#
# Phase 1 — Incident Detection:  Custom YOLO model + sliding-window gate.
# Phase 2 — Plate Extraction:    YOLO tracking + forensic upscaling + OCR.
# Phase 3 — Multi-Camera Tracking: Parallel thread-per-camera plate search.

import torch
import cv2
import numpy as np
import easyocr
import os
import base64
import requests
from ultralytics import YOLO
from collections import Counter, deque
from difflib import SequenceMatcher
import threading
from queue import Queue
from datetime import datetime

# Base URL of the Django backend. All results are posted to these endpoints.
DJANGO_BASE_URL    = "https://nonclimbable-muddledly-jami.ngrok-free.dev"
CREATE_INCIDENT_API = f"{DJANGO_BASE_URL}/api/incident/create/"
SEARCH_RESULT_API   = f"{DJANGO_BASE_URL}/api/search/result/"

# Video path for the main incident detection feed (Phase 1).
INCIDENT_VIDEO_PATH = "/kaggle/input/roadsurveillance-testdata/normal-street.mp4"

# Camera feeds scanned during Phase 3 (cross-camera vehicle tracking).
# Each entry maps a camera ID to its video file and a human-readable location.
CAMERA_FEEDS = {
    'cam1': {'path': "/kaggle/input/roadsurveillance-testdata/cam3.mp4", 'location': 'Main Street Intersection'},
    'cam2': {'path': "/kaggle/input/roadsurveillance-testdata/cam1.mp4", 'location': 'Highway Exit 42'},
    'cam3': {'path': "/kaggle/input/roadsurveillance-testdata/cam2.mp4", 'location': 'Downtown Plaza'}
}

# Path to the custom model trained to detect 'accident' and 'fight' classes.
# This is the output of the training cell above.
CUSTOM_MODEL_PATH   = '/kaggle/working/runs/detect/Surveillance_System/purified_tracker/weights/best.pt'

# General YOLOv8m model used only for vehicle detection during plate extraction
# and camera scanning (Phases 2 and 3). Using the larger 'm' variant here
# improves vehicle localisation accuracy without affecting incident detection.
TRACKING_MODEL_PATH = 'yolov8m.pt'

# Minimum YOLO confidence score to accept a detection from the custom model.
CONFIDENCE_THRESHOLD   = 0.60
# Minimum string-similarity ratio (0–1) to accept a plate match in Phase 3.
PLATE_MATCH_CONFIDENCE = 0.8
# Seconds before the confirmed incident frame to start scanning for plates.
# A 4-second window captures plates just before impact when they are still readable.
PRE_IMPACT_SECONDS     = 4
# Laplacian variance below this value means the crop is too blurry for OCR.
BLUR_THRESHOLD         = 80.0
# Pixel radius around detected vehicle centers; only cars within this zone
# are candidates for plate extraction (avoids scanning unrelated vehicles).
IMPACT_ZONE_RADIUS     = 120

# Sliding-window incident gate — mirrors final_production_test logic exactly.
# An incident is only confirmed when ACTIVATION_THRESHOLD or more frames
# within the last DETECTION_WINDOW_SIZE frames contain a positive detection.
# This eliminates single-frame false positives caused by motion blur or
# partial occlusions.
DETECTION_WINDOW_SIZE = 30   # Rolling window length (frames).
ACTIVATION_THRESHOLD  = 15   # Minimum positive frames needed to confirm (50% of window).

# Select GPU if available; fall back to CPU for inference.
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device.upper()}")

# Load models once at startup to avoid reloading them inside the pipeline loops.
print("Loading custom surveillance model...")
custom_model = YOLO(CUSTOM_MODEL_PATH).to(device)

print("Loading tracking model (for plate/camera work)...")
tracking_model = YOLO(TRACKING_MODEL_PATH).to(device)

# Initialise EasyOCR for English text with GPU acceleration if available.
reader = easyocr.Reader(['en'], gpu=(device == 'cuda'))

# Create output directories for storing evidence snapshots.
os.makedirs('forensics_reports', exist_ok=True)
os.makedirs('tracking_evidence', exist_ok=True)

# Thread-safe queue that Phase 3 camera threads push their results into.
# The main thread drains this queue after all threads finish.
results_queue = Queue()
# Lock used to serialise YOLO model loading across threads so two threads
# do not compete for GPU memory during initialisation.
thread_lock = threading.Lock()


# ── Utility / helper functions ────────────────────────────────────────────────

def image_to_base64(img):
    """Encode a BGR OpenCV image as a JPEG base64 string for JSON serialisation."""
    _, buffer = cv2.imencode('.jpg', img)
    return base64.b64encode(buffer).decode()

def forensic_upscale(img):
    """Upscale a small plate crop 3x and apply CLAHE contrast enhancement.

    Lanczos resampling preserves edge sharpness better than bilinear or
    bicubic at high scale factors. CLAHE (Contrast Limited Adaptive Histogram
    Equalisation) then improves local contrast so faint plate characters
    become readable by the OCR engine.

    Returns the enhanced greyscale image, or None if the input is empty.
    """
    if img.size == 0:
        return None
    img  = cv2.resize(img, None, fx=3.0, fy=3.0, interpolation=cv2.INTER_LANCZOS4)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=5.0, tileGridSize=(8, 8))
    return clahe.apply(gray)

def is_blurry(gray, threshold=BLUR_THRESHOLD):
    """Return True if the greyscale image is too blurry for reliable OCR.

    Uses the variance of the Laplacian as a focus measure. A low variance
    indicates a lack of edges, which corresponds to a blurry image.
    """
    return cv2.Laplacian(gray, cv2.CV_64F).var() < threshold

def clean_text(text):
    """Strip non-alphanumeric characters from OCR output and convert to uppercase."""
    return "".join(e for e in text if e.isalnum()).upper()

def similar(a, b):
    """Return a 0–1 similarity ratio between two strings using SequenceMatcher."""
    return SequenceMatcher(None, a, b).ratio()

def get_video_fps(path):
    """Read the frame rate of a video file. Falls back to 30 FPS if unavailable."""
    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    cap.release()
    return fps if fps > 0 else 30

def frame_to_timestamp(frame, fps):
    """Convert a frame index to a MM:SS string given the video's frame rate."""
    sec = frame / fps
    return f"{int(sec // 60):02d}:{int(sec % 60):02d}"

def center(box):
    """Return the centroid [x, y] of a bounding box given as [x1, y1, x2, y2]."""
    return np.array([(box[0] + box[2]) / 2, (box[1] + box[3]) / 2])

def distance(p1, p2):
    """Euclidean distance between two 2-D points."""
    return np.linalg.norm(np.array(p1) - np.array(p2))

def iou(box1, box2):
    """Intersection-over-Union of two bounding boxes [x1,y1,x2,y2].

    Returns a value in [0, 1]; 1 means perfect overlap, 0 means no overlap.
    """
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - inter
    return inter / union if union > 0 else 0


# ── Impact center extraction ──────────────────────────────────────────────────

def extract_impact_centers(frame):
    """Detect all cars in a frame and return their centre coordinates.

    Called once when an incident is confirmed (Phase 1). The returned
    centres define the spatial zone used in Phase 2 to focus plate
    extraction on vehicles actually involved in the incident, ignoring
    unrelated traffic in the background.

    Returns:
        list of np.array([cx, cy]) — one entry per detected car.
    """
    results = tracking_model.predict(frame, conf=0.4, verbose=False)
    centers = []
    if results[0].boxes is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy().astype(int)
        clss  = results[0].boxes.cls.cpu().numpy().astype(int)
        for box, cls in zip(boxes, clss):
            if results[0].names[cls] == 'car':
                centers.append(center(box))
    return centers


# ── Phase 3 worker: single-camera plate scan ──────────────────────────────────

def scan_camera_feed(cam_id, cam_info, target_plate):
    """Thread worker that searches one camera feed for a target license plate.

    For every frame the function:
      1. Runs YOLO with persistent tracking to locate cars.
      2. Crops the bottom 50% of each car box (where the plate sits).
      3. Forensically upscales and enhances the crop.
      4. Runs EasyOCR restricted to alphanumeric characters.
      5. Compares the result to the target plate using substring match
         or fuzzy similarity; exits as soon as a match is found.

    Results (match or None) are pushed onto the shared results_queue
    so the caller can collect them after all threads complete.
    """
    print(f"    [{cam_id}] Thread started - Scanning {cam_info['location']}...")

    clean_target = clean_text(target_plate)
    cap = cv2.VideoCapture(cam_info['path'])

    vehicle_found       = False
    detection_time      = None
    best_match_snapshot = None
    found_plate_text    = ""

    # Load a dedicated YOLO instance per thread. The thread_lock ensures
    # GPU memory allocation is serialised so threads do not race each other.
    with thread_lock:
        local_model = YOLO(TRACKING_MODEL_PATH).to(device)
    local_reader = easyocr.Reader(['en'], gpu=(device == 'cuda'))

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # persist=True keeps object IDs stable across frames so the same
        # vehicle is not re-scanned under a different ID.
        results = local_model.track(frame, persist=True, verbose=False)

        if results[0].boxes.id is not None:
            boxes = results[0].boxes.xyxy.cpu().numpy().astype(int)
            ids   = results[0].boxes.id.cpu().numpy().astype(int)
            clss  = results[0].boxes.cls.cpu().numpy().astype(int)

            for box, id, cls in zip(boxes, ids, clss):
                if results[0].names[cls] == 'car':
                    x1, y1, x2, y2 = box
                    h = y2 - y1
                    # Crop the lower half of the car bounding box — that is
                    # where the rear license plate is typically located.
                    bumper_crop = frame[int(y1 + h * 0.50):y2, x1:x2]

                    processed_plate = forensic_upscale(bumper_crop)

                    if processed_plate is not None:
                        # allowlist restricts OCR to plate-legal characters only,
                        # reducing the chance of mis-reading symbols as letters.
                        ocr_hits = local_reader.readtext(
                            processed_plate,
                            detail=0,
                            allowlist='0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ'
                        )

                        for text in ocr_hits:
                            detected_clean = clean_text(text)
                            # Accept the detection if the target plate is a substring
                            # of the OCR result (handles partial reads) OR if the
                            # two strings are sufficiently similar (handles OCR noise).
                            if (clean_target in detected_clean) or (similar(clean_target, detected_clean) > PLATE_MATCH_CONFIDENCE):
                                if not vehicle_found:
                                    vehicle_found       = True
                                    detection_time      = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                                    found_plate_text    = text
                                    best_match_snapshot = frame.copy()
                                    # Annotate the snapshot with a green bounding box
                                    # and detection timestamp for the evidence record.
                                    cv2.rectangle(best_match_snapshot, (x1, y1), (x2, y2), (0, 255, 0), 4)
                                    cv2.putText(best_match_snapshot, f"FOUND: {detection_time}",
                                                (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
                                    print(f"      [{cam_id}] MATCH FOUND at {detection_time}")
                                break
                        if vehicle_found:
                            break
            if vehicle_found:
                break
        if vehicle_found:
            break

    cap.release()

    # Build the result dict if a match was found; otherwise result stays None.
    result = None
    if vehicle_found:
        result = {
            'camera':       cam_id,
            'location':     cam_info['location'],
            'timestamp':    detection_time,
            'matched_text': found_plate_text,
            'snapshot':     best_match_snapshot
        }

    # Push the result onto the shared queue regardless of outcome
    # so the caller always receives exactly one result per camera thread.
    results_queue.put({'cam_id': cam_id, 'plate': target_plate, 'result': result})

    if not vehicle_found:
        print(f"      [{cam_id}] Scan complete - No match found")


# ══════════════════════════════════════════════════════════════════════════════
# PHASE 1: INCIDENT DETECTION — CUSTOM MODEL + SLIDING-WINDOW GATE
#
# Each frame is run through the custom YOLO model trained to detect
# 'accident' and 'fight' classes. Results are pushed into two fixed-size
# deques (one per class). A 1 means the class was detected in that frame;
# 0 means it was not.
#
# An incident is confirmed only when the rolling sum of a deque reaches
# ACTIVATION_THRESHOLD — requiring SUSTAINED detection across at least
# 15 consecutive frames within the 30-frame window (50% occupancy).
# This prevents a single noisy or blurry frame from triggering a false alert.
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 70)
print("PHASE 1: INTELLIGENT INCIDENT DETECTION (CUSTOM MODEL)")
print("=" * 70)
print(f"   Window: {DETECTION_WINDOW_SIZE} frames | Threshold: {ACTIVATION_THRESHOLD} positive frames")

cap = cv2.VideoCapture(INCIDENT_VIDEO_PATH)
fps = get_video_fps(INCIDENT_VIDEO_PATH)

incident_snapshot = None   # Frame saved at the moment of incident confirmation.
incident_frame    = 0      # Frame index at which the incident was confirmed.
impact_centers    = []     # Vehicle centroids used to scope Phase 2 plate extraction.
incident_type     = None   # 'accident', 'fight', or None if nothing was found.

# Independent sliding windows, one per incident class.
recent_accidents = deque(maxlen=DETECTION_WINDOW_SIZE)
recent_fights    = deque(maxlen=DETECTION_WINDOW_SIZE)

frame_idx = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    frame_idx += 1

    timestamp = frame_idx / fps if fps > 0 else 0

    # Run the custom surveillance model on this frame.
    results       = custom_model.predict(frame, conf=CONFIDENCE_THRESHOLD, verbose=False)
    frame_classes = [
        custom_model.names[int(box.cls[0].item())]
        for box in results[0].boxes
    ]

    is_acc = 'accident' in frame_classes
    is_fgt = 'fight'    in frame_classes

    # Append binary signals to the rolling windows.
    recent_accidents.append(1 if is_acc else 0)
    recent_fights.append(1 if is_fgt else 0)

    acc_score = sum(recent_accidents)
    fgt_score = sum(recent_fights)

    status_text = "ACCIDENT" if is_acc else ("FIGHT" if is_fgt else "NORMAL")
    print(
        f"Frame {frame_idx:04d} | {timestamp:.2f}s | "
        f"Detection: {status_text:<10} | "
        f"Acc window: {acc_score:02d}/{ACTIVATION_THRESHOLD} | "
        f"Fight window: {fgt_score:02d}/{ACTIVATION_THRESHOLD}"
    )

    # Check whether the rolling sum has crossed the activation threshold.
    # On confirmation, capture the snapshot and vehicle positions, then break.
    if acc_score >= ACTIVATION_THRESHOLD:
        print(f"\nACCIDENT CONFIRMED AT FRAME {frame_idx}")
        print(f"   Sustained detections: {acc_score}/{DETECTION_WINDOW_SIZE} frames")
        incident_type     = 'accident'
        incident_snapshot = frame.copy()
        incident_frame    = frame_idx
        # Capture vehicle centroids at the moment of confirmation to define
        # the impact zone used in Phase 2 plate extraction.
        impact_centers    = extract_impact_centers(frame)
        break

    elif fgt_score >= ACTIVATION_THRESHOLD:
        print(f"\nFIGHT CONFIRMED AT FRAME {frame_idx}")
        print(f"   Sustained detections: {fgt_score}/{DETECTION_WINDOW_SIZE} frames")
        incident_type     = 'fight'
        incident_snapshot = frame.copy()
        incident_frame    = frame_idx
        break

cap.release()

if incident_type is None:
    print("\nNo incidents detected in video footage.")


# ══════════════════════════════════════════════════════════════════════════════
# PHASE 2: FORENSIC PLATE EXTRACTION
#
# Only runs when Phase 1 confirms an 'accident' (fights have no vehicles
# to track). The pipeline re-opens the incident video and scans only the
# PRE_IMPACT_SECONDS window before the confirmed incident frame — that
# short window is where the involved vehicles are still moving and their
# plates are most likely to be visible and unobstructed.
#
# Plate selection strategy:
#   1. Prefer plates that match the standard 6-char format (3 letters + 3 digits).
#   2. If none match, fall back to prefix-based scoring (most common 3-char
#      prefix + letter count + character-position frequency).
# ══════════════════════════════════════════════════════════════════════════════
detected_plates = []

if incident_type == 'accident':
    print(f"\nPHASE 2: FORENSIC PLATE EXTRACTION")
    print("=" * 70)

    PRE_IMPACT_FRAMES = int(fps * PRE_IMPACT_SECONDS)
    start_frame       = max(0, incident_frame - PRE_IMPACT_FRAMES)
    end_frame         = incident_frame

    print(f"Scanning frames {start_frame} to {end_frame}  (PRE-IMPACT WINDOW)")

    cap          = cv2.VideoCapture(INCIDENT_VIDEO_PATH)
    plate_ballot = []   # All OCR plate readings collected across the window.

    # Debug counters help diagnose where in the pipeline readings are lost.
    debug_stats = {
        "frames_scanned":  0,
        "cars_seen":       0,
        "cars_near_impact": 0,
        "bumper_crops":    0,
        "ocr_runs":        0,
        "ocr_texts":       0
    }

    frame_idx = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame_idx += 1
        if frame_idx < start_frame:
            continue
        if frame_idx > end_frame:
            break

        # Use persistent tracking so each vehicle keeps the same ID across
        # frames, avoiding redundant OCR on the same vehicle.
        results = tracking_model.track(frame, persist=True, conf=0.4, verbose=False)

        if results[0].boxes.id is None:
            continue

        boxes = results[0].boxes.xyxy.cpu().numpy().astype(int)
        ids   = results[0].boxes.id.cpu().numpy().astype(int)
        clss  = results[0].boxes.cls.cpu().numpy().astype(int)

        debug_stats["frames_scanned"] += 1

        for box, id, cls in zip(boxes, ids, clss):
            if results[0].names[cls] == 'car':
                debug_stats["cars_seen"] += 1

                cx = (box[0] + box[2]) // 2
                cy = (box[1] + box[3]) // 2

                # Only examine cars whose centre falls within the impact zone.
                # This filters out unrelated vehicles passing in the background.
                if impact_centers and not any(
                    np.linalg.norm(np.array([cx, cy]) - np.array(ic)) < IMPACT_ZONE_RADIUS
                    for ic in impact_centers
                ):
                    continue

                debug_stats["cars_near_impact"] += 1

                x1, y1, x2, y2 = box
                h = y2 - y1
                # Crop the bottom half of the bounding box with a small margin
                # to avoid clipping the plate edge.
                bumper_crop = frame[
                    int(y1 + h * 0.50): y2 + 5,
                    max(0, x1 - 5): min(frame.shape[1], x2 + 5)
                ]
                debug_stats["bumper_crops"] += 1

                processed = forensic_upscale(bumper_crop)
                if processed is None:
                    continue

                # Discard blurry crops — OCR on motion-blurred plates produces
                # unreliable results that pollute the plate ballot.
                if is_blurry(processed):
                    continue

                ocr_hits = reader.readtext(
                    processed,
                    detail=0,
                    allowlist='0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ'
                )

                debug_stats["ocr_runs"] += 1

                for text in ocr_hits:
                    detected_clean = clean_text(text)

                    # Strip a leading '0' or '1' that OCR frequently misreads
                    # as a prefix when the plate starts with a letter O or I.
                    if len(detected_clean) > 6 and detected_clean[0] in "01":
                        detected_clean = detected_clean[1:]

                    # Only add readings long enough to be a valid plate (≥6 chars).
                    if len(detected_clean) >= 6:
                        plate_ballot.append(detected_clean)
                        debug_stats["ocr_texts"] += 1

    cap.release()

    # Print the debug report so it is easy to identify bottlenecks.
    print("\nFORENSIC DEBUG REPORT")
    print("────────────────────────")
    print(f"Frames scanned           : {debug_stats['frames_scanned']}")
    print(f"Cars detected            : {debug_stats['cars_seen']}")
    print(f"Cars near impact zone    : {debug_stats['cars_near_impact']}")
    print(f"Bumper crops extracted   : {debug_stats['bumper_crops']}")
    print(f"OCR runs executed        : {debug_stats['ocr_runs']}")
    print(f"OCR text hits            : {debug_stats['ocr_texts']}")

    if plate_ballot:
        print(f"\nPlate ballot: {plate_ballot}")

        def count_letters_and_digits(text):
            """Return (letter_count, digit_count) for a string."""
            letters = sum(1 for c in text if c.isalpha())
            digits  = sum(1 for c in text if c.isdigit())
            return letters, digits

        # Step 1: filter for plates matching the standard 6-char format (3L+3D).
        # This is the most reliable reading; prefer it over any fallback result.
        valid_plates = []
        for plate in plate_ballot:
            if len(plate) == 6:
                letters, digits = count_letters_and_digits(plate)
                if letters == 3 and digits == 3:
                    valid_plates.append(plate)

        print(f"Plates matching 3L+3D pattern: {valid_plates}")

        if valid_plates:
            # Pick the most frequently appearing valid plate as the consensus answer.
            valid_counter   = Counter(valid_plates)
            most_common     = valid_counter.most_common(1)[0]
            detected_plates = [most_common[0]]
            print(f"PLATES IDENTIFIED: {detected_plates}")
            print(f"   Frequency: {most_common[1]} occurrences")
        else:
            # Step 2: fallback scoring when no plate matches the standard format.
            # Group readings by their first 3 characters (most common prefix),
            # then score each unique candidate by:
            #   - letter count * 10  (more letters = more plate-like)
            #   - occurrence count * 5 (appears more often = more reliable)
            #   - per-position character frequency (consistent chars = correct read)
            print("\nNo plates match standard 3L+3D pattern — attempting fallback...")

            prefixes = [plate[:3] for plate in plate_ballot if len(plate) >= 3]
            if prefixes:
                prefix_counter     = Counter(prefixes)
                most_common_prefix = prefix_counter.most_common(1)[0][0]
                # Restrict candidates to plates sharing the most common prefix.
                prefix_plates = [p for p in plate_ballot if p.startswith(most_common_prefix)]

                # Build a character-position frequency map across all prefix plates.
                # A character that appears consistently at the same position is more
                # likely to be correctly read.
                char_pos_freq = {}
                for plate in prefix_plates:
                    for pos, char in enumerate(plate):
                        key = (pos, char)
                        char_pos_freq[key] = char_pos_freq.get(key, 0) + 1

                unique_plates = list(set(prefix_plates))
                scored_plates = []

                for plate in unique_plates:
                    letter_count     = sum(1 for c in plate if c.isalpha())
                    occurrence_count = prefix_plates.count(plate)
                    freq_score       = sum(char_pos_freq.get((pos, char), 0) for pos, char in enumerate(plate))
                    total_score      = (letter_count * 10) + (occurrence_count * 5) + freq_score
                    scored_plates.append((plate, total_score, letter_count, occurrence_count))

                # Sort descending — highest score is the most credible plate reading.
                scored_plates.sort(key=lambda x: x[1], reverse=True)

                if scored_plates:
                    best_plate      = scored_plates[0]
                    detected_plates = [best_plate[0]]
                    print(f"FALLBACK PLATE IDENTIFIED: {detected_plates}")
                    print(f"   Letters: {best_plate[2]}, Occurrences: {best_plate[3]}, Score: {best_plate[1]}")
                else:
                    print("\nNo plates detected in pre-impact frames")
            else:
                print("\nNo plates detected in pre-impact frames")
    else:
        print("\nNo plates detected in pre-impact frames")


# ══════════════════════════════════════════════════════════════════════════════
# PHASE 3: PARALLEL MULTI-CAMERA VEHICLE TRACKING
#
# Only runs when Phase 2 identifies at least one license plate.
# For each detected plate, one thread is spawned per camera feed.
# All threads run simultaneously so the scan of N cameras takes roughly
# the same wall-clock time as scanning a single camera.
# Results are collected from the shared queue after all threads join.
# ══════════════════════════════════════════════════════════════════════════════
vehicle_routes = {}

if incident_type == 'accident' and detected_plates:
    print("\n" + "=" * 70)
    print("PHASE 3: PARALLEL MULTI-CAMERA TRACKING SYSTEM")
    print("=" * 70)

    vehicle_routes = {plate: [] for plate in detected_plates}

    for target_plate in detected_plates:
        print(f"\nTracking plate: {target_plate} — Scanning {len(CAMERA_FEEDS)} cameras...")

        # Launch one thread per camera — all scan in parallel.
        threads = []
        for cam_id, cam_info in CAMERA_FEEDS.items():
            thread = threading.Thread(
                target=scan_camera_feed,
                args=(cam_id, cam_info, target_plate)
            )
            threads.append(thread)
            thread.start()

        # Wait for all camera threads to finish before collecting results.
        for thread in threads:
            thread.join()

        print(f"   All {len(CAMERA_FEEDS)} camera scans completed for {target_plate}")

        # Drain the shared queue and save matched snapshots as evidence.
        while not results_queue.empty():
            result_data = results_queue.get()
            if result_data['plate'] == target_plate and result_data['result'] is not None:
                detection  = result_data['result']
                vehicle_routes[target_plate].append(detection)
                save_path  = f"tracking_evidence/{target_plate}_{result_data['cam_id']}.jpg"
                cv2.imwrite(save_path, detection['snapshot'])


# ══════════════════════════════════════════════════════════════════════════════
# DJANGO API PAYLOAD CONSTRUCTION AND SUBMISSION
#
# Assembles the complete incident record (type, snapshot, vehicles, detections)
# into a JSON payload and POSTs it to the Django backend.
# On network failure the payload is saved locally as a JSON backup so the
# evidence is not lost and can be submitted manually later.
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("SENDING INCIDENT DATA TO DJANGO API")
print("=" * 70)

if incident_type:
    # Save the incident snapshot to disk as forensic evidence.
    snapshot_path = f"forensics_reports/{incident_type}_incident.jpg"
    cv2.imwrite(snapshot_path, incident_snapshot)

    payload = {
        "incident_type": incident_type,
        "location":      "Main Feed",
        "timestamp":     datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "snapshot":      image_to_base64(incident_snapshot),
        "vehicles":      []
    }

    # Attach vehicle and detection data only for accidents (fights have no plates).
    if incident_type == 'accident' and detected_plates:
        for plate in detected_plates:
            vehicle_data = {
                "plate":      plate,
                "confidence": 0.9,
                "detections": []
            }

            if plate in vehicle_routes:
                # Sort detections chronologically so Django displays the route
                # in the order the vehicle was actually sighted.
                detections = vehicle_routes[plate]
                detections.sort(key=lambda x: x['timestamp'])
                for detection in detections:
                    vehicle_data["detections"].append({
                        "camera":       detection['camera'],
                        "location":     detection['location'],
                        "timestamp":    detection['timestamp'],
                        "matched_text": detection['matched_text']
                    })

            payload["vehicles"].append(vehicle_data)

    try:
        print(f"Posting to: {CREATE_INCIDENT_API}")
        response = requests.post(CREATE_INCIDENT_API, json=payload, timeout=10)

        if response.status_code in (200, 201):
            print("Incident successfully sent to Django backend")
            print(f"Response: {response.json()}")
        else:
            print(f"API returned status code: {response.status_code}")
            print(f"Response: {response.text}")

    except requests.exceptions.RequestException as e:
        print(f"Failed to send to Django API: {e}")
        # Save the payload locally as a fallback so the data is not lost
        # and can be submitted to the API manually once connectivity is restored.
        print("Payload saved locally for manual upload")
        import json
        with open('forensics_reports/api_payload_backup.json', 'w') as f:
            backup_payload            = payload.copy()
            # Strip the large base64 snapshot from the local backup to keep
            # the file human-readable; the full snapshot is already on disk.
            backup_payload['snapshot'] = '[BASE64_DATA_REMOVED]'
            json.dump(backup_payload, f, indent=2)


# ══════════════════════════════════════════════════════════════════════════════
# FINAL CONSOLE REPORT
#
# Summarises the outcome of all three phases in a single block so the full
# result is easy to read at a glance without scrolling through frame logs.
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("FINAL COMPREHENSIVE REPORT")
print("=" * 70)

if incident_type:
    print(f"""
INCIDENT SUMMARY:
- Type      : {incident_type.upper()}
- Time      : {frame_to_timestamp(incident_frame, fps)}  (Frame: {incident_frame})
- Location  : Main Feed
- Detection : Custom Model (sustained {ACTIVATION_THRESHOLD}/{DETECTION_WINDOW_SIZE} frames)
""")

    if incident_type == 'accident':
        if detected_plates:
            print(f"- Plates Identified : {', '.join(detected_plates)}")
            print(f"\nTRACKING RESULTS:")
            print(f"- Cameras Scanned   : {len(CAMERA_FEEDS)} (Parallel)")

            for plate in detected_plates:
                print(f"\nVehicle: {plate}")
                if plate in vehicle_routes and vehicle_routes[plate]:
                    detections = vehicle_routes[plate]
                    # Build a human-readable route string showing the camera sequence.
                    route = " -> ".join([d['camera'] for d in detections])
                    print(f"   Route   : {route}")
                    print(f"   Cameras : {len(detections)}/{len(CAMERA_FEEDS)}")
                    print(f"   Timeline:")
                    for d in detections:
                        print(f"      - {d['camera']}: {d['timestamp']} @ {d['location']}")
                else:
                    print(f"   Status: NOT TRACKED")
        else:
            print("- Plates: NOT VISIBLE")
            print("\nVehicle tracking skipped — no readable plates")

    print(f"\nEVIDENCE SAVED:")
    print(f"- Snapshot : {snapshot_path}")
    if vehicle_routes:
        print(f"- Tracking : tracking_evidence/")
else:
    print("No incidents detected in the video footage.")

print("\n" + "=" * 70)
print("INTELLIGENT INCIDENT ANALYSIS COMPLETE")
print("=" * 70)
